# KrishiNexus · `yolov11n-krishi-bd-v2.1`: potato quality vision on Kaggle

This notebook trains the two-stage potato inspector behind the KrishiNexus optical grader. It then scores each stage
once on the dataset's own test split.

- **Stage 1, potato finder:** a YOLO11n-seg model trained on the ROI set (2,386 market photos, one class). It outlines
  each potato, so the app can cut the potato out of the background the way the defect crops were cut.
- **Stage 2, defect segmenter:** trained on the defect set (1,724 background-removed tuber crops, with four classes:
  scab, rot, greening and mechanical damage).
  - Run 1, YOLO11n-seg at 640 px, reached validation mask mAP50 0.396 and missed its 0.50 target.
  - This run trains two candidates, YOLO11n-seg at 1024 px and YOLO11s-seg at 640 px, and keeps the one with the
    higher **validation** mask mAP50.
- **Grading check:** on each test crop, the predicted defect masks become the grader's "bruising %" (scab, greening and
  mechanical damage) and "rot %". The app's BARI grade rule turns these into grade A, B or C. That grade is compared
  with the grade from the hand-drawn labels, and with a majority-grade baseline taken from the train split.

**Data.** Tasnia, M. H. (2026). *Diamant Potato Dataset for ROI and Surface Defect Instance Segmentation*. Mendeley
Data, V1. doi:10.17632/4vty9rz7zw.1. Daffodil International University, CC BY 4.0. The photos were taken at the
Ashulia, Banasree and Khilgaon markets in Dhaka.
- The notebook downloads both zips (8.9 GB) and checks each against its published SHA-256.
- It also checks the image and instance counts against the dataset page before training.

**Before you run**
1. **Settings → Accelerator → GPU T4 ×2.**
   - The two GPUs train side by side: the 1024 px candidate on one, the potato finder and then the larger model on
     the other. One GPU also works, sequentially.
   - Without a GPU the notebook runs a **CPU smoke test**: 1 epoch on 5% of the train split at 320 px.
   - The smoke test checks that every cell works. Its scores are not results, and its report says so.
2. **Settings → Internet → on.** The data, `pip` and the YOLO11 starting weights are downloaded.

**Time (estimated from run 1):** about 10 min for the data, then about 2 h of training on T4 ×2.

**Frozen-test rule:** training, checkpoint selection and the choice between the candidates read only `train` and
`val`. Section 5 opens `test` once, for the winner only. Never tune anything against it.

**Outputs (`/kaggle/working`):**
- `yolov11n-krishi-bd-v2.1/`: `roi_best.pt`, `defect_best.pt`, the two data YAMLs, `eval_report.json`, `run_manifest.json`
- `runs/`: Ultralytics training curves, plots and test-split plots
- `data_checks.json`: what was found in the zips

**Licences:** the dataset is CC BY 4.0, so credit its authors. Ultralytics YOLO11 code and weights are AGPL-3.0.


## 1. Config


In [ ]:
import os
os.environ["WANDB_MODE"] = "disabled"

SEED = 20261109
WORK = "/kaggle/working"
RAW = "/tmp/diamant_raw"            # downloads and extracted zips (not kept)
DATA = "/tmp/diamant_yolo"          # resized YOLO dataset (not kept)
OUT = f"{WORK}/yolov11n-krishi-bd-v2.1"
RUNS = f"{WORK}/runs"
MAX_SIDE = 1024                     # stored image size; each model resizes again to its own imgsz
# Stage 1 outlines each potato, so the app can cut it out of the market background as the defect crops were cut.
ROI = dict(weights="yolo11n-seg.pt", imgsz=640, epochs=80, patience=20, batch=32)
# Run 1 (YOLO11n-seg at 640 px) reached validation mask mAP50 0.396. Two stronger candidates, chosen between on val only:
DEFECT_CANDIDATES = {"n1024": dict(weights="yolo11n-seg.pt", imgsz=1024, epochs=150, patience=40, batch=16),
                     "s640": dict(weights="yolo11s-seg.pt", imgsz=640, epochs=150, patience=40, batch=16)}
CONF = 0.25                         # detection threshold for the grading check

ZIPS = {   # Mendeley Data file links and their published SHA-256
    "defect": ("https://data.mendeley.com/public-files/datasets/4vty9rz7zw/files/4863d50a-390b-474b-9ef8-482a42b7bc33/file_downloaded",
               "89cbdfc1072415e35ec11fc4a22318a848fa90285b1a927112438af1c1bc3d48"),
    "roi": ("https://data.mendeley.com/public-files/datasets/4vty9rz7zw/files/187b5ef3-d642-4cbd-9cef-3dcab2d84ddd/file_downloaded",
            "27a32167f8c71b88c8ec88f80249b91a3a019ecc1992ed6e95f67fac262cea0d"),
}
PUBLISHED = {   # from the dataset page: images per split, and instances (per class, per split for the defect set)
    # The page's "2,581 annotated potato instances" is the train split's count; val and test add 645 and 363.
    "roi": {"images": {"train": 1670, "val": 477, "test": 239}, "train_instances": 2581},
    "defect": {"images": {"train": 1281, "val": 176, "test": 267},
               "instances": {"train": [3188, 903, 79, 562], "val": [548, 175, 17, 108], "test": [654, 190, 29, 123]}},
}
TARGETS = {"roi_box_mAP50": 0.90, "defect_mask_mAP50": 0.50, "grade_accuracy": 0.70}   # set before the first run (TRAINING_PLAN.md section 8)


## 2. Helpers (self-tested in the repo)


In [ ]:
"""Helpers for the Diamant potato vision notebook: dataset layout, YOLO labels and the grading metrics."""
import pathlib

import numpy as np

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}
SPLIT_NAMES = {"train": "train", "training": "train", "val": "val", "valid": "val", "validation": "val",
               "test": "test", "testing": "test"}
DEFECT_NAMES = ["scab", "rot", "greening", "mechanical_damage"]   # class ids 0-3, as published on the dataset page
BLEMISH, ROT = (0, 2, 3), (1,)   # the grader's two inputs: surface blemish feeds "bruising %", decay feeds "rot %"
GRADE_LIMITS = {"a_bruising": 3, "a_rot": 2, "b_bruising": 8, "b_rot": 5}   # prompt_registry "bari-standard"


def split_of(path):
    """The split named by the nearest directory in the path, or None."""
    for part in reversed(pathlib.PurePath(path).parts[:-1]):
        if part.lower() in SPLIT_NAMES:
            return SPLIT_NAMES[part.lower()]
    return None


def label_path(image):
    """YOLO convention: swap the nearest 'images' directory for 'labels' and use a .txt suffix."""
    parts = list(pathlib.PurePath(image).parts)
    for i in range(len(parts) - 2, -1, -1):
        if parts[i].lower() == "images":
            parts[i] = "labels"
            break
    return pathlib.Path(*parts).with_suffix(".txt")


def pair(files):
    """{split: [(image, label path or None)]} for every image, skipping macOS resource forks."""
    labels = {str(f) for f in files if f.suffix.lower() == ".txt"}
    out = {}
    for f in files:
        if f.suffix.lower() in IMAGE_EXTS and "__MACOSX" not in f.parts and not f.name.startswith("._"):
            lab = label_path(f)
            out.setdefault(split_of(f), []).append((f, lab if str(lab) in labels else None))
    return out


def read_label(text):
    """[(class_id, coords)] from a YOLO label: 4 numbers are a box, an even number above 4 a polygon."""
    rows = []
    for line in text.splitlines():
        tok = line.split()
        if not tok:
            continue
        nums = [float(t) for t in tok[1:]]
        if len(nums) < 4 or (len(nums) > 4 and len(nums) % 2):
            raise ValueError(f"bad YOLO label line: {line[:80]}")
        rows.append((int(float(tok[0])), np.array(nums, dtype=np.float32)))
    return rows


def polygon_mask(rows, classes, h, w):
    """Union mask of the labelled shapes whose class is in `classes`; boxes are drawn as rectangles."""
    from PIL import Image, ImageDraw

    img = Image.new("1", (w, h), 0)
    draw = ImageDraw.Draw(img)
    for cls, c in rows:
        if cls not in classes:
            continue
        if len(c) == 4:
            x, y, bw, bh = c
            c = np.array([x - bw / 2, y - bh / 2, x + bw / 2, y - bh / 2, x + bw / 2, y + bh / 2, x - bw / 2, y + bh / 2])
        pts = [(float(px) * w, float(py) * h) for px, py in c.reshape(-1, 2)]
        if len(pts) >= 3:
            draw.polygon(pts, fill=1)
    return np.array(img, dtype=bool)


def union(masks, cls, classes, shape):
    """Union of the predicted masks whose class is in `classes` (an empty mask when there are none)."""
    keep = np.isin(cls, classes)
    return masks[keep].any(axis=0) if keep.any() else np.zeros(shape, dtype=bool)


def area_pct(mask, potato):
    """Share of the potato's visible pixels that the mask covers, in percent."""
    n = int(potato.sum())
    return 100.0 * float((mask & potato).sum()) / n if n else 0.0


def grade(bruising, rot, lim=GRADE_LIMITS):
    """The app's grade rule for one potato (uniformity is a lot-level measure, so it is left out)."""
    if bruising <= lim["a_bruising"] and rot <= lim["a_rot"]:
        return "A"
    if bruising <= lim["b_bruising"] and rot <= lim["b_rot"]:
        return "B"
    return "C"


def presence(gt_sets, pred_sets, n_classes):
    """Per class, precision/recall/F1 of the image-level call "this potato shows class c"."""
    out = {}
    for c in range(n_classes):
        tp = sum(c in g and c in p for g, p in zip(gt_sets, pred_sets))
        fp = sum(c not in g and c in p for g, p in zip(gt_sets, pred_sets))
        fn = sum(c in g and c not in p for g, p in zip(gt_sets, pred_sets))
        prec = tp / (tp + fp) if tp + fp else None
        rec = tp / (tp + fn) if tp + fn else None
        f1 = None if prec is None or rec is None else (2 * prec * rec / (prec + rec) if prec + rec else 0.0)
        out[c] = {"precision": prec, "recall": rec, "f1": f1, "support": tp + fn}
    return out


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics"], check=True)
import torch, ultralytics
GPUS = torch.cuda.device_count()
GPU = GPUS > 0
SMOKE = not GPU                     # CPU session: 1 epoch on 5% of train at 320 px, to exercise every cell; not a result
DEVICE = 0 if GPU else "cpu"
size = lambda spec: 320 if SMOKE else spec["imgsz"]   # the image size a model trains and is scored at
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "|",
      f"{GPUS} x {torch.cuda.get_device_name(0)}" if GPU else "no GPU: CPU smoke test")
try:
    from ultralytics import settings
    settings.update({"wandb": False, "mlflow": False})
except Exception as e:   # the settings keys differ between Ultralytics versions; logging integrations are optional
    print("settings not changed:", e)
from ultralytics.utils.downloads import attempt_download_asset
for weights in ("yolo11n-seg.pt", "yolo11s-seg.pt"):   # fetched up front: GitHub releases sometimes answer 504 for minutes
    attempt_download_asset(weights, retry=8)


## 3. Data: download, verify, convert


In [ ]:
import hashlib, shutil, time, zipfile

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 23), b""):
            h.update(block)
    return h.hexdigest()

pathlib.Path(RAW).mkdir(parents=True, exist_ok=True)
print("free disk, GB: /tmp", shutil.disk_usage("/tmp").free // 2**30, "| /kaggle/working", shutil.disk_usage(WORK).free // 2**30)
t0 = time.time()
for name, (url, digest) in ZIPS.items():
    zpath = pathlib.Path(RAW, f"{name}.zip")
    subprocess.run(["curl", "-L", "--fail", "--silent", "--show-error", "--retry", "5", "--retry-delay", "15",
                    "-o", str(zpath), url], check=True)
    got = sha256(zpath)
    assert got == digest, f"{name}.zip has sha256 {got}, but the dataset page publishes {digest}"
    with zipfile.ZipFile(zpath) as z:
        z.extractall(pathlib.Path(RAW, name))
    zpath.unlink()
    print(f"{name}: downloaded, sha256 verified, extracted ({time.time() - t0:.0f} s)")


In [ ]:
import json
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from PIL import Image, ImageOps
import yaml

def show_layout(root):
    files = sorted(p for p in pathlib.Path(root).rglob("*") if p.is_file())
    tree = Counter((str(p.parent.relative_to(root)), p.suffix.lower()) for p in files)
    for (d, ext), n in sorted(tree.items()):
        print(f"  {n:6d}  {d}/*{ext}")
    for p in files:
        if p.suffix.lower() in (".yaml", ".yml") or p.name.lower().startswith("readme"):
            print(f"  --- {p.relative_to(root)}\n" + p.read_text(errors="replace")[:1500])
    return files

def convert(job):
    name, split, img, lab = job
    im = Image.open(img)
    oriented = im.getexif().get(0x0112, 1) != 1
    if im.format == "JPEG":
        im.draft("RGB", (MAX_SIDE, MAX_SIDE))          # decode 3000 px JPEGs at a reduced scale
    im = ImageOps.exif_transpose(im).convert("RGB")    # cv2, and so Ultralytics, applies EXIF orientation as well
    im.thumbnail((MAX_SIDE, MAX_SIDE), Image.BILINEAR)
    jpg = img.suffix.lower() in (".jpg", ".jpeg")
    dst = pathlib.Path(DATA, name, "images", split, img.stem + (".jpg" if jpg else ".png"))
    im.save(dst, quality=92) if jpg else im.save(dst, compress_level=1)   # defect crops stay lossless
    text = lab.read_text() if lab else ""
    pathlib.Path(DATA, name, "labels", split, img.stem + ".txt").write_text(text)
    return split, read_label(text), oriented, lab is None

checks = {}
for name, names in (("roi", ["potato"]), ("defect", DEFECT_NAMES)):
    print(f"== {name} dataset layout")
    pairs = pair(show_layout(pathlib.Path(RAW, name)))
    assert None not in pairs, f"{len(pairs[None])} {name} images sit outside a train/val/test folder (see the layout above)"
    for split, items in pairs.items():
        assert len({img.stem for img, _ in items}) == len(items), f"duplicate file names in {name}/{split}"
        for sub in ("images", "labels"):
            pathlib.Path(DATA, name, sub, split).mkdir(parents=True, exist_ok=True)
    with ThreadPoolExecutor(8) as ex:
        done = list(ex.map(convert, [(name, s, img, lab) for s, items in pairs.items() for img, lab in items]))
    per_class = Counter((s, cls) for s, rows, _, _ in done for cls, _ in rows)
    coords = [c for _, rows, _, _ in done for _, c in rows]
    flat = np.concatenate(coords) if coords else np.zeros(1)
    checks[name] = {
        "images": dict(Counter(s for s, *_ in done)),
        "instances": {s: [per_class[(s, c)] for c in range(len(names))] for s in pairs},
        "unknown_class_ids": sorted({c for _, c in per_class} - set(range(len(names)))),
        "shapes": dict(Counter("box" if len(c) == 4 else "polygon" for c in coords)),
        "missing_label_files": sum(m for *_, m in done),
        "exif_rotated_images": sum(o for _, _, o, _ in done),
        "coord_range": [round(float(flat.min()), 4), round(float(flat.max()), 4)],
    }
    print(json.dumps(checks[name]))
    pathlib.Path(DATA, f"{name}.yaml").write_text(yaml.safe_dump(
        {"path": f"{DATA}/{name}", "train": "images/train", "val": "images/val", "test": "images/test", "names": dict(enumerate(names))}))
    shutil.rmtree(pathlib.Path(RAW, name))

problems = []
for name in ("roi", "defect"):
    if checks[name]["images"] != PUBLISHED[name]["images"]:
        problems.append(f"{name} images per split {checks[name]['images']} differ from the published {PUBLISHED[name]['images']}")
    if checks[name]["unknown_class_ids"]:
        problems.append(f"{name} labels use unknown class ids {checks[name]['unknown_class_ids']}")
if checks["roi"]["instances"]["train"] != [PUBLISHED["roi"]["train_instances"]]:
    problems.append(f"roi train instances {checks['roi']['instances']['train']} differ from the published {PUBLISHED['roi']['train_instances']}")
if checks["defect"]["instances"] != PUBLISHED["defect"]["instances"]:
    problems.append(f"defect instances {checks['defect']['instances']} differ from the published {PUBLISHED['defect']['instances']}")
if checks["defect"]["shapes"].get("box"):
    problems.append("defect labels contain boxes; the segmenter needs polygons")
checks["problems"] = problems
json.dump(checks, open(f"{WORK}/data_checks.json", "w"), indent=2)
print("\n".join(problems) or "Counts match the dataset page: images per split, and instances per class and split.")
if GPU:
    assert not problems, "the data does not match the dataset page; see data_checks.json"


## 4. Train the potato finder and both defect candidates (train + val only)


In [ ]:
from ultralytics import YOLO

JOBS = pathlib.Path("/tmp/train_jobs.py")          # one process per GPU, each training its queue in order
JOBS.write_text("import json, sys\nfrom ultralytics import YOLO\nfor job in json.loads(sys.argv[1]):\n"
                "    YOLO(job.pop('weights')).train(**job)\n")

def job(name, spec, data):
    run = (dict(epochs=1, fraction=0.05, batch=8, cache=False) if SMOKE else
           dict(epochs=spec["epochs"], patience=spec["patience"], batch=spec["batch"], cache="ram"))
    return dict(weights=spec["weights"], data=data, imgsz=size(spec), seed=SEED, deterministic=True, workers=2,
                project=RUNS, name=name, exist_ok=True, plots=True, verbose=False, **run)

roi_job = job("roi", ROI, f"{DATA}/roi.yaml")
cand_jobs = {k: job(f"defect_{k}", spec, f"{DATA}/defect.yaml") for k, spec in DEFECT_CANDIDATES.items()}
# With two T4s, GPU 0 trains the 1024 px candidate while GPU 1 trains the potato finder and then the larger model.
queues = {0: [cand_jobs["n1024"]], 1: [roi_job, cand_jobs["s640"]]} if GPUS >= 2 else {DEVICE: [roi_job, *cand_jobs.values()]}
started, procs = time.time(), {}
for device, jobs in queues.items():
    for j in jobs:
        j["device"] = device
    procs[device] = subprocess.Popen([sys.executable, str(JOBS), json.dumps(jobs)],
                                     stdout=open(f"{WORK}/train_{device}.log", "w"), stderr=subprocess.STDOUT)
for device, p in procs.items():
    print(f"device {device} exited {p.wait()}; log tail:\n" + pathlib.Path(f"{WORK}/train_{device}.log").read_text(errors="replace")[-1200:])
timing = {"train_wall_minutes": round((time.time() - started) / 60, 1)}

def best(name):
    path = pathlib.Path(RUNS, name, "weights", "best.pt")
    return str(path) if path.exists() else None

pathlib.Path(OUT).mkdir(parents=True, exist_ok=True)
ROI_BEST = best("roi")
if ROI_BEST:                                       # kept even if a later cell fails
    shutil.copy(ROI_BEST, f"{OUT}/roi_best.pt")
    shutil.copy(f"{DATA}/roi.yaml", f"{OUT}/roi.yaml")
CANDIDATES = {k: best(f"defect_{k}") for k in DEFECT_CANDIDATES}
print("trained:", {"roi": ROI_BEST, **CANDIDATES})


## 5. Choose the defect model on validation, then score once on the frozen test splits


In [ ]:
import traceback
import cv2

results = {}

def stage(metrics, kind):
    d = metrics.results_dict
    return {k: round(float(d[f"metrics/{k}({kind})"]), 4) for k in ("precision", "recall", "mAP50", "mAP50-95")}

def by_class(metrics, ap50):
    return {metrics.names[int(c)]: round(float(a), 4) for c, a in zip(metrics.ap_class_index, ap50)}

def gt_areas(img_path, split):
    potato = np.array(Image.open(img_path).convert("RGB")).max(axis=2) > 10   # crops sit on uniform black
    rows = read_label(pathlib.Path(DATA, "defect/labels", split, pathlib.Path(img_path).stem + ".txt").read_text())
    h, w = potato.shape
    return rows, potato, [area_pct(polygon_mask(rows, g, h, w), potato) for g in (BLEMISH, ROT)]

VAL, CHOSEN = {}, None
try:   # the candidates are compared on the validation split only
    for k, weights in CANDIDATES.items():
        if weights:
            m = YOLO(weights).val(data=f"{DATA}/defect.yaml", split="val", imgsz=size(DEFECT_CANDIDATES[k]), batch=16,
                                  device=DEVICE, project=RUNS, name=f"defect_{k}_val", exist_ok=True, verbose=False)
            VAL[k] = {"mask": stage(m, "M"), "box": stage(m, "B")}
    CHOSEN = max(VAL, key=lambda k: VAL[k]["mask"]["mAP50"])
    DEFECT_BEST, DEFECT_IMGSZ = CANDIDATES[CHOSEN], size(DEFECT_CANDIDATES[CHOSEN])
    shutil.copy(DEFECT_BEST, f"{OUT}/defect_best.pt")
    shutil.copy(f"{DATA}/defect.yaml", f"{OUT}/defect.yaml")
    print("validation mask mAP50:", {k: v["mask"]["mAP50"] for k, v in VAL.items()}, "-> chosen:", CHOSEN)
except Exception:
    results["selection_error"] = traceback.format_exc()
    print(results["selection_error"])
try:   # run 1 opened the test split for its 640 px nano model; this run opens it once more, for the winner only
    roi = YOLO(ROI_BEST).val(data=f"{DATA}/roi.yaml", split="test", imgsz=size(ROI), batch=32, device=DEVICE,
                             project=RUNS, name="roi_test", exist_ok=True, verbose=False)
    results["roi_test"] = {"box": stage(roi, "B"), "mask": stage(roi, "M"), "n_images": PUBLISHED["roi"]["images"]["test"]}
    dfc = YOLO(DEFECT_BEST).val(data=f"{DATA}/defect.yaml", split="test", imgsz=DEFECT_IMGSZ, batch=16, device=DEVICE,
                                project=RUNS, name="defect_test", exist_ok=True, verbose=False)
    results["defect_test"] = {"box": stage(dfc, "B"), "mask": stage(dfc, "M"), "n_images": PUBLISHED["defect"]["images"]["test"],
                              "mask_ap50_by_class": by_class(dfc, dfc.seg.ap50), "box_ap50_by_class": by_class(dfc, dfc.box.ap50)}
except Exception:
    results["test_error"] = traceback.format_exc()
    print(results["test_error"])
try:
    train_grades = Counter(grade(*gt_areas(p, "train")[2]) for p in sorted(pathlib.Path(DATA, "defect/images/train").iterdir()))
    majority = train_grades.most_common(1)[0][0]
    model = YOLO(DEFECT_BEST)
    test_imgs = sorted(str(p) for p in pathlib.Path(DATA, "defect/images/test").iterdir())
    gt_sets, pred_sets, grades, err_b, err_r = [], [], [], [], []
    for res in model.predict(test_imgs, imgsz=DEFECT_IMGSZ, conf=CONF, retina_masks=True, stream=True, device=DEVICE, verbose=False):
        rows, potato, (gb, gr) = gt_areas(res.path, "test")
        h, w = potato.shape
        cls = res.boxes.cls.cpu().numpy().astype(int)
        masks = res.masks.data.cpu().numpy().astype(np.uint8) if res.masks is not None else np.zeros((0, h, w), np.uint8)
        if masks.shape[1:] != (h, w):
            masks = np.stack([cv2.resize(m, (w, h), interpolation=cv2.INTER_NEAREST) for m in masks]) if len(masks) else np.zeros((0, h, w), np.uint8)
        masks = masks.astype(bool)
        pb, pr = (area_pct(union(masks, cls, g, (h, w)), potato) for g in (BLEMISH, ROT))
        gt_sets.append({c for c, _ in rows})
        pred_sets.append(set(cls.tolist()))
        grades.append((grade(gb, gr), grade(pb, pr)))
        err_b.append(abs(pb - gb))
        err_r.append(abs(pr - gr))
    n = len(grades)
    rnd = lambda x: round(x, 4) if isinstance(x, float) else x
    recalls = [sum(p == g for gg, p in grades if gg == g) / k for g in "ABC" if (k := sum(gg == g for gg, _ in grades))]
    results["grading"] = {
        "n_potatoes": n, "conf": CONF,
        "grade_accuracy": round(sum(g == p for g, p in grades) / n, 4),
        "grade_balanced_accuracy": round(sum(recalls) / len(recalls), 4),   # mean per-grade recall: always "C" scores 0.33
        "majority_grade_baseline": {"grade": majority, "accuracy": round(sum(g == majority for g, _ in grades) / n, 4),
                                    "train_grade_counts": dict(train_grades)},
        "confusion_label_to_predicted": {g: dict(Counter(p for gg, p in grades if gg == g)) for g in "ABC"},
        "bruising_pct_mae": round(float(np.mean(err_b)), 2), "rot_pct_mae": round(float(np.mean(err_r)), 2),
        "presence": {DEFECT_NAMES[c]: {k: rnd(v) for k, v in d.items()} for c, d in presence(gt_sets, pred_sets, len(DEFECT_NAMES)).items()},
    }
except Exception:
    results["grading_error"] = traceback.format_exc()
    print(results["grading_error"])
print(json.dumps(results, indent=1)[:6000])


## 6. Report and manifest


In [ ]:
import platform
import PIL
pathlib.Path(OUT).mkdir(parents=True, exist_ok=True)
values = {"roi_box_mAP50": results.get("roi_test", {}).get("box", {}).get("mAP50"),
          "defect_mask_mAP50": results.get("defect_test", {}).get("mask", {}).get("mAP50"),
          "grade_accuracy": results.get("grading", {}).get("grade_accuracy")}
epochs_run = {name: max(0, len(pathlib.Path(RUNS, name, "results.csv").read_text().splitlines()) - 1)
              for name in ("roi", *(f"defect_{k}" for k in DEFECT_CANDIDATES)) if pathlib.Path(RUNS, name, "results.csv").exists()}
chosen = DEFECT_CANDIDATES.get(CHOSEN, {})
report = {
    "model": "yolov11n-krishi-bd-v2.1",
    "status": "trained" if GPU else "cpu_smoke_test: 1 epoch on 5% of train at 320 px; these scores are not results",
    "stages": {"roi": f"{ROI['weights']} at {size(ROI)} px, 1 class (potato), outlines",
               "defect": f"{chosen.get('weights')} at {size(chosen) if chosen else None} px, 4 classes " + ", ".join(DEFECT_NAMES)},
    "selection": {"by": "validation mask mAP50", "validation": VAL, "chosen": CHOSEN},
    "dataset": {"name": "Diamant Potato Dataset", "doi": "10.17632/4vty9rz7zw.1", "licence": "CC BY 4.0",
                "sha256": {k: v[1] for k, v in ZIPS.items()}, "checks": checks},
    "grade_inputs": {"bruising_pct": [DEFECT_NAMES[c] for c in BLEMISH], "rot_pct": [DEFECT_NAMES[c] for c in ROT],
                     "limits": GRADE_LIMITS},
    "targets": TARGETS,
    "summary": [{"metric": k, "target": TARGETS[k], "value": v, "meets_target": None if v is None else v >= TARGETS[k]} for k, v in values.items()],
    "results": results,
}
manifest = {
    "seed": SEED, "max_side": MAX_SIDE, "roi": ROI, "defect_candidates": DEFECT_CANDIDATES, "chosen": CHOSEN,
    "epochs_run": epochs_run, "timing": globals().get("timing", {}),
    "gpu": f"{GPUS} x {torch.cuda.get_device_name(0)}" if GPU else None, "python": platform.python_version(),
    "versions": {"ultralytics": ultralytics.__version__, "torch": torch.__version__, "numpy": np.__version__,
                 "pillow": PIL.__version__, "opencv": cv2.__version__},
}
json.dump(report, open(f"{OUT}/eval_report.json", "w"), indent=2)
json.dump(manifest, open(f"{OUT}/run_manifest.json", "w"), indent=2)
for row in report["summary"]:
    print(f"{row['metric']:<20} target {row['target']:<5} value {row['value']}  meets: {row['meets_target']}")
